# **Task 3: Linked spatial visualization**

In [11]:
import os
import sys
import json
from pathlib import Path


import pandas as pd
import numpy as np
import altair as alt


import geopandas as gpd
from shapely.geometry import Point

# Show all rows
pd.set_option('display.max_rows', None)

# Show all columns
pd.set_option('display.max_columns', None)
#pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)
#pd.set_option("display.max_colwidth", 200)

print("Versions ->",
      "pandas:", pd.__version__,
      "| geopandas:", gpd.__version__)

Versions -> pandas: 2.2.2 | geopandas: 1.1.1


In [12]:
from google.colab import drive
drive.mount('/content/drive/')

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [14]:
file_path = '/content/drive/MyDrive/CS424-Assignment3/df_sub.pkl'
df = pd.read_pickle(file_path)


bad_id = "S00542058-I1"

before = len(df)
df = df[df["Job Filing Number"] != bad_id].copy()
df.to_pickle("/content/drive/MyDrive/CS424-Assignment3/df_sub.pkl")
print(f"Removed {before - len(df)} rows; new shape: {df.shape}")
df.shape

Removed 1 rows; new shape: (679969, 81)


(679969, 81)

In [15]:
#  Building a GeoDataFrame from Longitude/Latitude & sanity-check

# Ensure required columns exist
assert {"Longitude", "Latitude"}.issubset(df.columns), "Longitude/Latitude missing in df_ready."

# Create geometry in WGS84 (EPSG:4326)
gdf_ready = gpd.GeoDataFrame(
    df.copy(),
    geometry=gpd.points_from_xy(df["Longitude"], df["Latitude"]),
    crs="EPSG:4326"
)

print("GeoDataFrame created.")
print(f"Shape: {gdf_ready.shape}")
print(f"CRS:   {gdf_ready.crs}")

# Spatial bounds check (NYC-ish bbox expected)
minx, miny, maxx, maxy = gdf_ready.total_bounds
print("Total bounds (lon/lat):")
print(f" - lon_min: {minx:.6f}, lon_max: {maxx:.6f}")
print(f" - lat_min: {miny:.6f}, lat_max: {maxy:.6f}")

# Quick borough distribution snapshot (top 10)
if "Borough" in gdf_ready.columns:
    print("\nTop borough counts:")
    display(gdf_ready["Borough"].value_counts().head(10).to_frame("count"))

# Peek a few rows with geometry
gdf_ready[["Borough", "Job Type", "Filing Status", "Latitude", "Longitude", "geometry"]].head(3)


GeoDataFrame created.
Shape: (679969, 82)
CRS:   EPSG:4326
Total bounds (lon/lat):
 - lon_min: -74.254967, lon_max: -73.700384
 - lat_min: 40.498966, lat_max: 40.912886

Top borough counts:


,count
Borough,
MANHATTAN,261398
BROOKLYN,186311
QUEENS,133838
BRONX,65216
STATEN ISLAND,33206


,Borough,Job Type,Filing Status,Latitude,Longitude,geometry
0,BROOKLYN,Alteration,Permit Entire,40.661309,-74.000422,POINT (-74.00042 40.66131)
1,BROOKLYN,Alteration,Permit Entire,40.661309,-74.000422,POINT (-74.00042 40.66131)
2,BROOKLYN,Full Demolition,Pending Plan Examiner Assignment,40.685604,-73.926614,POINT (-73.92661 40.6856)


In [16]:
# to check that GeoDataFrame is present & in sync with FINAL_DF (single step) ---
FINAL_DF = df.copy()
# 1) Create FINAL_GDF if it doesn't exist yet
if 'FINAL_GDF' not in globals():
    assert {'Longitude', 'Latitude'}.issubset(FINAL_DF.columns), "Missing lon/lat in FINAL_DF."
    FINAL_GDF = gpd.GeoDataFrame(
        FINAL_DF.copy(),
        geometry=gpd.points_from_xy(FINAL_DF['Longitude'], FINAL_DF['Latitude']),
        crs='EPSG:4326'
    )
    created = True
else:
    created = False
    # 2) Light sync: ensure non-geometry columns match dtypes/values
    #    (geometry is left untouched)
    missing_in_gdf = [c for c in FINAL_DF.columns if c not in FINAL_GDF.columns]
    for c in missing_in_gdf:
        FINAL_GDF[c] = FINAL_DF[c]

    common = [c for c in FINAL_DF.columns if c in FINAL_GDF.columns and c != 'geometry']
    # If dtype differs, overwrite GDF column with DF version to keep types aligned
    for c in common:
        if FINAL_GDF[c].dtype != FINAL_DF[c].dtype:
            FINAL_GDF[c] = FINAL_DF[c]

    # Ensure CRS is set
    if FINAL_GDF.crs is None:
        FINAL_GDF.set_crs('EPSG:4326', inplace=True)

# 3) Report
print("GeoDataFrame status:")
print(" - Created now?" , created)
print(" - Shape (FINAL_DF): ", FINAL_DF.shape)
print(" - Shape (FINAL_GDF):", FINAL_GDF.shape)
print(" - CRS:", FINAL_GDF.crs)

# 4) Quick column parity check
cols_df  = set(FINAL_DF.columns)
cols_gdf = set(FINAL_GDF.columns) - {'geometry'}
print("\nColumns missing in FINAL_GDF:", sorted(cols_df - cols_gdf) if (cols_df - cols_gdf) else "None")
print("Extra non-geometry columns in FINAL_GDF:", sorted(cols_gdf - cols_df) if (cols_gdf - cols_df) else "None")


GeoDataFrame status:
 - Created now? False
 - Shape (FINAL_DF):  (679969, 81)
 - Shape (FINAL_GDF): (679968, 82)
 - CRS: EPSG:4326

Columns missing in FINAL_GDF: None
Extra non-geometry columns in FINAL_GDF: None


In [17]:
# --- Choropleth + param-bubbles + linked time series ---



alt.data_transformers.disable_max_rows()

# -------------------- INPUTS --------------------

NTA_GEOJSON_PATH = Path("/content/drive/MyDrive/CS424-Assignment3/nyc_nta.geojson")   # your NTA polygons



# Ensure we have Filing Date + NTA + Job Type + unique filing id
pts = (
    FINAL_GDF[["Filing Date", "NTA", "Job Type", "Job Filing Number"]]
    .dropna(subset=["Filing Date", "NTA", "Job Type"])
    .copy()
)
pts["NTA_key"] = pts["NTA"].astype('string').str.strip().str.upper()
pts["month_start"] = pts["Filing Date"].values.astype("datetime64[M]")
pts["mmYYYY"] = pts["month_start"].dt.strftime("%Y-%m")

# -------------------- POLYGONS & STATIC CHOROPLETH DATA --------------------
gpoly = gpd.read_file(NTA_GEOJSON_PATH)

candidate_cols = [
    'NTAName','ntaName','ntaname',
    'NTA','nta',
    'NTA2020','NTA2020_NAME','NTAName20',
    'NTACode','NTACode20','NTACode_20',
    'ntacode','ntacode20'
]
poly_key_col = next((c for c in candidate_cols if c in gpoly.columns), None)
if poly_key_col is None:
    raise ValueError(
        f"Could not find an NTA name/code column in polygons. "
        f"Available columns: {list(gpoly.columns)}\n"
        f"Tip: rename the proper column to 'NTAName' and re-run."
    )

gpoly = gpoly.copy()
gpoly["NTA_key"] = gpoly[poly_key_col].astype("string").str.strip().str.upper()
gpoly["NTA_label"] = gpoly[poly_key_col].astype("string")

# Static 2021–2025 total filings per NTA for base choropleth color
nta_totals = (
    pts.assign(yr=pts["month_start"].dt.year)
       .query("yr >= 2021 and yr <= 2025")
       .groupby("NTA_key", observed=False)["Job Filing Number"]
       .nunique()
       .rename("filings_total")
       .reset_index()
)

# Merge totals into polygons
gpoly2 = gpoly.merge(nta_totals, on="NTA_key", how="left")
gpoly2["filings_total"] = gpoly2["filings_total"].fillna(0)

# Ensure WGS84
if gpoly2.crs is None or gpoly2.crs.to_epsg() != 4326:
    gpoly2 = gpoly2.to_crs(epsg=4326)

# Robust centroids: compute in projected CRS, then back to 4326
gpoly_proj = gpoly2.to_crs(epsg=2263)  # NAD83 / New York Long Island (ftUS)
cent_geom_proj = gpoly_proj.geometry.centroid
cent_geom_wgs = gpd.GeoSeries(cent_geom_proj, crs=2263).to_crs(epsg=4326)
centroids_df = pd.DataFrame({
    "NTA_key": gpoly2["NTA_key"].values,
    "lon": cent_geom_wgs.x.values,
    "lat": cent_geom_wgs.y.values
})


nta_monthly = (
    pts.groupby(["NTA_key","Job Type","month_start","mmYYYY"], observed=False)["Job Filing Number"]
       .nunique()
       .rename("filings")
       .reset_index()
)

city_monthly = (
    pts.groupby(["month_start","mmYYYY","Job Type"], observed=False)["Job Filing Number"]
       .nunique()
       .rename("filings_city")
       .reset_index()
)

jobtype_opts = sorted(pts["Job Type"].dropna().astype(str).unique().tolist())

month_order = (
    city_monthly[["month_start","mmYYYY"]]
    .drop_duplicates()
    .sort_values("month_start")["mmYYYY"]
    .tolist()
)

# -------------------- INTERACTIVE PARAMS --------------------
p_job = alt.param(
    name="jobSel", value="(All)",
    bind=alt.binding_select(options=["(All)"] + jobtype_opts, name="Job Type")
)
brush = alt.selection_interval(encodings=["x"])
nta_click = alt.selection_point(fields=["NTA_key"])

# -------------------- CHOROPLETH (symlog only) --------------------
gjson = gpoly2.to_json()

choropleth = (
    alt.Chart(alt.Data(values=gjson, format={'type': 'json', 'property': 'features'}))
      .transform_calculate(NTA_key="datum.properties.NTA_key")
      .mark_geoshape(stroke='black', strokeWidth=0.25)
      .encode(
          color=alt.Color(
              'properties.filings_total:Q',
              title='Total filings 2021–2025',
              scale=alt.Scale(type='symlog', scheme='viridis')
          ),
          tooltip=[
              alt.Tooltip(f'properties.{poly_key_col}:N', title='NTA'),
              alt.Tooltip('properties.filings_total:Q', title='Total filings', format=','),
          ],
          opacity=alt.condition(nta_click, alt.value(0.95), alt.value(0.75))
      )
      .add_params(nta_click)
      .properties(width=520, height=620, title="Filings by NTA (symlog)")
)

# -------------------- BUBBLES (counts within brushed window) --------------------
bubbles = (
    alt.Chart(nta_monthly)
      .transform_calculate(passJob="(isValid(jobSel) && jobSel != '(All)') ? datum['Job Type'] == jobSel : true")
      .transform_filter("datum.passJob")
      .transform_filter(brush)  # only the brushed months
      .transform_aggregate(total_filings="sum(filings)", groupby=["NTA_key"])
      .transform_lookup(
          lookup="NTA_key",
          from_=alt.LookupData(centroids_df, key="NTA_key", fields=["lon","lat"])
      )
      .mark_circle(stroke="black", strokeWidth=0.5)
      .encode(
          longitude="lon:Q",
          latitude="lat:Q",
          size=alt.Size("total_filings:Q", title="Filings (in brush)", scale=alt.Scale(range=[10, 1200])),
          color=alt.Color("total_filings:Q", title="Filings (in brush)", scale=alt.Scale(scheme="plasma")),
          tooltip=[
              alt.Tooltip("NTA_key:N", title="NTA key"),
              alt.Tooltip("total_filings:Q", title="Filings (window)", format=","),
          ],
          opacity=alt.condition(nta_click, alt.value(1.0), alt.value(0.9))
      )
      .properties(width=520, height=620)
)

map_comp = alt.layer(choropleth, bubbles).resolve_scale(color="independent", size="independent")

# -------------------- LINKED TIME SERIES --------------------
ts_base = alt.Chart(nta_monthly).properties(width=900, height=180)

ts_sel = (
    ts_base
    .transform_filter(nta_click)
    .transform_calculate(passJob="(isValid(jobSel) && jobSel != '(All)') ? datum['Job Type'] == jobSel : true")
    .transform_filter("datum.passJob")
    .transform_aggregate(filings="sum(filings)", groupby=["mmYYYY","month_start"])
    .mark_line(point=True)
    .encode(
        x=alt.X("mmYYYY:N", sort=month_order, title="Month"),
        y=alt.Y("filings:Q", title="Filings (selected NTA)"),
        tooltip=[alt.Tooltip("mmYYYY:N", title="Month"),
                 alt.Tooltip("filings:Q", title="Filings", format=",")]
    )
)

ts_city = (
    alt.Chart(city_monthly)
    .transform_calculate(passJob="(isValid(jobSel) && jobSel != '(All)') ? datum['Job Type'] == jobSel : true")
    .transform_filter("datum.passJob")
    .transform_aggregate(filings_city="sum(filings_city)", groupby=["mmYYYY","month_start"])
    .mark_area(opacity=0.25)
    .encode(
        x=alt.X("mmYYYY:N", sort=month_order, title="Month (brush to set map window)"),
        y=alt.Y("filings_city:Q", title="Citywide filings"),
        tooltip=[alt.Tooltip("mmYYYY:N", title="Month"),
                 alt.Tooltip("filings_city:Q", title="Citywide", format=",")]
    )
    .add_params(brush)
    .properties(width=900, height=120)
)

ts_title = (
    alt.Chart(pd.DataFrame({'t': ['Monthly filings — select an NTA on map; brush to aggregate bubbles']}))
      .mark_text(dy=-5, fontSize=13, fontWeight='bold', align='left')
      .encode(text='t:N')
      .properties(width=900, height=20)
)

ts_comp = alt.vconcat(ts_title, ts_city, ts_sel).resolve_scale(x="shared")


final_map_dashboard = alt.hconcat(
    map_comp,
    ts_comp
).add_params(p_job).properties(
    title="NTA Filings — Symlog Choropleth + Bubble Overlay (Job-type & Brush Filtered) with Linked Time Series"
)

final_map_dashboard


Output hidden; open in https://colab.research.google.com to view.

In [18]:
# === TASK 3: LINKED SPATIAL (Task 2) + NON-SPATIAL (Task 1) DASHBOARD ===
# Links: Choropleth + Bubbles + Time Series (Task 2)
#    with: (A) Monthly clustered bars, (B) Work-type mix, (C) Boro×JobType heatmap (Task 1 Viz1),
#          Butterfly + Monthly diverging bars (Task 1 Viz3),
#          Ridgeline + Top-10 (Task 1 Viz4)
import altair as alt, pandas as pd, numpy as np
import geopandas as gpd
from pathlib import Path

# ---- Altair setup ----
alt.data_transformers.disable_max_rows()
alt.renderers.enable("default")

# -------------------------- SET ME --------------------------
DF = FINAL_DF.copy()  # your cleaned DF

# Robust path: prefer Drive path if present, fallback to /mnt/data
p_colab = Path("/content/drive/MyDrive/CS424-Assignment3/nyc_nta.geojson")
p_local = Path("/mnt/data/nyc_nta.geojson")
NTA_GEOJSON_PATH = p_colab if p_colab.exists() else p_local
# ------------------------------------------------------------

# ==================== COMMON PREP ====================
# Dates
for c in ["Filing Date","Approved Date","First Permit Date","Signoff Date"]:
    if c in DF.columns:
        DF[c] = pd.to_datetime(DF[c], errors="coerce")

DF = DF.dropna(subset=["Filing Date","Borough","Job Type"]).copy()
DF["year"]        = DF["Filing Date"].dt.year.astype(int)
DF["month_num"]   = DF["Filing Date"].dt.month.astype(int)
DF["month_lbl"]   = DF["Filing Date"].dt.strftime("%b")
DF["month_start"] = DF["Filing Date"].values.astype("datetime64[M]")
DF["mmYYYY"]      = DF["month_start"].dt.strftime("%Y-%m")

# Borough normalization
boro_map = {
    "bronx":"Bronx","brooklyn":"Brooklyn","bk":"Brooklyn",
    "manhattan":"Manhattan","mn":"Manhattan",
    "queens":"Queens","qn":"Queens",
    "staten island":"Staten Island","si":"Staten Island"
}
DF["Borough"] = (
    DF["Borough"].astype(str).str.strip()
      .map(lambda s: boro_map.get(s.lower(), s.title()))
)

BORO_DOMAIN = ["Manhattan","Brooklyn","Queens","Bronx","Staten Island"]

# NTA → Borough (dominant borough by your DF)
nta_boro = (
    DF.dropna(subset=["NTA","Borough"])[["NTA","Borough"]]
      .assign(NTA_key=lambda d: d["NTA"].astype("string").str.strip().str.upper())
      .groupby("NTA_key", observed=False)["Borough"]
      .agg(lambda s: s.value_counts().idxmax())
      .reset_index()
)

# ==================== SPATIAL (Task 2 base) ====================
gpoly = gpd.read_file(NTA_GEOJSON_PATH)

candidate_cols = [
    'NTAName','ntaName','ntaname',
    'NTA','nta',
    'NTA2020','NTA2020_NAME','NTAName20',
    'NTACode','NTACode20','NTACode_20',
    'ntacode','ntacode20'
]
poly_key_col = next((c for c in candidate_cols if c in gpoly.columns), None)
if poly_key_col is None:
    raise ValueError(f"No NTA name/code column found in polygons. Columns: {list(gpoly.columns)}")

gpoly = gpoly.copy()
gpoly["NTA_key"]   = gpoly[poly_key_col].astype("string").str.strip().str.upper()
gpoly["NTA_label"] = gpoly[poly_key_col].astype("string")

# Total filings per NTA (2021–2025) for choropleth color
nta_totals = (
    DF.loc[(DF["year"]>=2021) & (DF["year"]<=2025)]
      .assign(NTA_key=lambda d: d["NTA"].astype("string").str.strip().str.upper())
      .groupby("NTA_key", observed=False)["Job Filing Number"]
      .nunique().rename("filings_total").reset_index()
)
gpoly2 = gpoly.merge(nta_totals, on="NTA_key", how="left").fillna({"filings_total":0})

# CRS + centroids for bubbles
if gpoly2.crs is None or (getattr(gpoly2.crs,"to_epsg",lambda:None)() != 4326):
    gpoly2 = gpoly2.to_crs(epsg=4326)
gpoly_proj = gpoly2.to_crs(epsg=2263)
cent_geom_proj = gpoly_proj.geometry.centroid
cent_wgs = gpd.GeoSeries(cent_geom_proj, crs=2263).to_crs(epsg=4326)
centroids_df = pd.DataFrame({
    "NTA_key": gpoly2["NTA_key"],
    "lon": cent_wgs.x.values,
    "lat": cent_wgs.y.values
})

# Monthly per NTA & city
nta_monthly = (
    DF.assign(NTA_key=lambda d: d["NTA"].astype("string").str.strip().str.upper())
      .groupby(["NTA_key","Job Type","month_start","mmYYYY"], observed=False)["Job Filing Number"]
      .nunique().rename("filings").reset_index()
)
city_monthly = (
    DF.groupby(["month_start","mmYYYY","Job Type"], observed=False)["Job Filing Number"]
      .nunique().rename("filings_city").reset_index()
)
month_order = (
    city_monthly[["month_start","mmYYYY"]].drop_duplicates()
      .sort_values("month_start")["mmYYYY"].tolist()
)

# ==================== SHARED CONTROLS/SELECTIONS ====================
jobtype_opts = ["(All)"] + sorted(DF["Job Type"].dropna().astype(str).unique().tolist())
jobSel = alt.param(name="jobSel", value="(All)",
                   bind=alt.binding_select(options=jobtype_opts, name="Job Type"))
boroughSel  = alt.selection_point(name="boroughSel", fields=["Borough"], empty=True)
brushMonths = alt.selection_interval(name="brushMonths", encodings=["x"], empty=True)

# Helper expr (applies Job Type filter where needed)
job_pass_expr = "(isValid(jobSel) && jobSel != '(All)') ? datum['Job Type'] == jobSel : true"

# ==================== MAP (Task 2, borough-linked) ====================
gjson = gpoly2.to_json()

choropleth = (
    alt.Chart(alt.Data(values=gjson, format={'type':'json','property':'features'}))
      .transform_calculate(NTA_key="datum.properties.NTA_key")
      .transform_lookup(
          lookup="NTA_key",
          from_=alt.LookupData(nta_boro, key="NTA_key", fields=["Borough"])
      )
      .mark_geoshape(stroke='black', strokeWidth=0.25)
      .encode(
          color=alt.Color('properties.filings_total:Q',
                          title='Total filings 2021–2025',
                          scale=alt.Scale(type='symlog', scheme='viridis')),
          tooltip=[
              alt.Tooltip(f'properties.{poly_key_col}:N', title='NTA'),
              alt.Tooltip('Borough:N', title='Borough'),
              alt.Tooltip('properties.filings_total:Q', title='Total filings', format=','),
          ],
          opacity=alt.condition(boroughSel, alt.value(1.0), alt.value(0.85))
      )
      .add_params(boroughSel)
      .properties(width=520, height=620, title="Filings by NTA (symlog); click to select Borough")
)

bubbles = (
    alt.Chart(nta_monthly)
      .transform_filter(job_pass_expr)
      .transform_filter(brushMonths)
      .transform_aggregate(total_filings="sum(filings)", groupby=["NTA_key"])
      .transform_lookup(
          lookup="NTA_key",
          from_=alt.LookupData(centroids_df, key="NTA_key", fields=["lon","lat"])
      )
      .transform_lookup(
          lookup="NTA_key",
          from_=alt.LookupData(nta_boro, key="NTA_key", fields=["Borough"])
      )
      .transform_filter("!length(data('boroughSel_store').values) || boroughSel")
      .mark_circle(stroke="black", strokeWidth=0.5)
      .encode(
          longitude="lon:Q", latitude="lat:Q",
          size=alt.Size("total_filings:Q", title="Filings (in brush)", scale=alt.Scale(range=[10, 1200])),
          color=alt.Color("total_filings:Q", title="Filings (in brush)", scale=alt.Scale(scheme="plasma")),
          tooltip=[alt.Tooltip("Borough:N"), alt.Tooltip("total_filings:Q", title="Filings (window)", format=",")],
          opacity=alt.condition(boroughSel, alt.value(1.0), alt.value(0.9))
      )
      .properties(width=520, height=620)
)

map_comp = alt.layer(choropleth, bubbles).resolve_scale(color="independent", size="independent")

# ==================== TIME SERIES (Task 2) ====================
ts_city = (
    alt.Chart(city_monthly)
      .transform_filter(job_pass_expr)
      .transform_aggregate(filings_city="sum(filings_city)", groupby=["mmYYYY","month_start"])
      .mark_area(opacity=0.25)
      .encode(
          x=alt.X("mmYYYY:N", sort=month_order, title="Month (brush to set window)"),
          y=alt.Y("filings_city:Q", title="Citywide filings"),
          tooltip=[alt.Tooltip("mmYYYY:N", title="Month"), alt.Tooltip("filings_city:Q", title="Citywide", format=",")]
      )
      .add_params(brushMonths)
      .properties(width=900, height=120)
)

boro_monthly = (
    DF.groupby(["Borough","month_start","mmYYYY","Job Type"], observed=False)["Job Filing Number"]
      .nunique().rename("filings").reset_index()
)

ts_boro = (
    alt.Chart(boro_monthly)
      .transform_filter(job_pass_expr)
      .transform_filter("!length(data('boroughSel_store').values) || boroughSel")
      .transform_aggregate(filings="sum(filings)", groupby=["mmYYYY","month_start"])
      .mark_line(point=True)
      .encode(
          x=alt.X("mmYYYY:N", sort=month_order, title="Month"),
          y=alt.Y("filings:Q", title="Filings (selected Boroughs)"),
          tooltip=[alt.Tooltip("mmYYYY:N", title="Month"), alt.Tooltip("filings:Q", title="Filings", format=",")]
      )
      .properties(width=900, height=180)
)

ts_title = (
    alt.Chart(pd.DataFrame({'t': ['Monthly filings — select Boroughs on map; brush to aggregate bubbles + charts']}))
      .mark_text(dy=-5, fontSize=13, fontWeight='bold', align='left')
      .encode(text='t:N')
      .properties(width=900, height=20)
)

ts_comp = alt.vconcat(ts_title, ts_city, ts_boro).resolve_scale(x="shared")

# ==================== TASK 1 Viz 1 (A/B/C) — LINKED ====================
m_job = (
    DF.groupby(["year","month_num","month_lbl","mmYYYY","Job Type","Borough"], observed=False)
      .size().reset_index(name="filings")
)

topN = 6
top_jobs = (
    m_job.groupby("Job Type", observed=False)["filings"].sum()
         .sort_values(ascending=False).head(topN).index.tolist()
)

legend_sel = alt.selection_point(fields=["Job Type"], bind="legend", toggle=True, empty=True)

axis_month = alt.Axis(
    title="Month",
    labelAngle=-40,
    labelExpr='["","Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"][toNumber(datum.value)]'
)

bars = (
    alt.Chart(m_job)
      .transform_filter(brushMonths)
      .transform_filter(legend_sel)
      .transform_filter("!length(data('boroughSel_store').values) || boroughSel")
      .transform_aggregate(filings="sum(filings)", groupby=["year","month_num","month_lbl","Job Type","mmYYYY"])
      .mark_bar()
      .encode(
          x=alt.X("month_num:O", sort=list(range(1,13)), axis=axis_month),
          y=alt.Y("filings:Q", title="Filings (count)"),
          color=alt.Color("Job Type:N", title="Job Type", scale=alt.Scale(domain=top_jobs)),
          xOffset=alt.XOffset("Job Type:N"),
          order=alt.Order("Job Type:N"),
          tooltip=["month_lbl:N","Job Type:N",alt.Tooltip("filings:Q", title="Filings")]
      )
      .add_params(legend_sel)
      .properties(title="(A) Monthly Filings by Job Type — Linked to Map Brush & Borough Selection",
                  width=900, height=320)
)

# Work-type mix
worktype_cols = [c for c in DF.columns if c.endswith("( Work Type)") or c.endswith("(Work Type)")]
def to_bool_series(s):
    if s.dtype == bool: return s.fillna(False)
    if pd.api.types.is_numeric_dtype(s): return s.fillna(0).astype(int).astype(bool)
    sv = s.astype(str).str.strip().str.lower(); return sv.isin(["y","yes","true","t","1"])
base_wt = DF[["Borough","Job Type","mmYYYY"] + worktype_cols].copy()
for c in worktype_cols: base_wt[c] = to_bool_series(base_wt[c])
longWT = base_wt.melt(id_vars=["Borough","Job Type","mmYYYY"], value_vars=worktype_cols,
                      var_name="WorkTypeRaw", value_name="has_work")
longWT = longWT[longWT["has_work"]].copy()
longWT["WorkType"] = longWT["WorkTypeRaw"].str.replace(r"\s*\( ?Work Type ?\)\s*", "", regex=True)

worktype_mix = (
    alt.Chart(longWT)
      .transform_filter(brushMonths)
      .transform_filter(job_pass_expr)
      .transform_filter("!length(data('boroughSel_store').values) || boroughSel")
      .mark_bar()
      .encode(
          y=alt.Y("Borough:N", sort=BORO_DOMAIN, title="Borough"),
          x=alt.X("count():Q", stack="normalize", title="Share of filings"),
          color=alt.Color("WorkType:N", title="Work Type"),
          tooltip=[alt.Tooltip("Borough:N"), alt.Tooltip("WorkType:N", title="Work Type"),
                   alt.Tooltip("count():Q", title="Filings", format=",.0f")]
      )
      .properties(title="(B) Work-Type Composition by Borough — Linked", width=440, height=220)
)

# Heatmap (Borough × Job Type)
monthly_jobtype_boro = (
    DF.groupby(["mmYYYY","Borough","Job Type"], observed=False)
      .size().reset_index(name="filings")
)

heatmap_C = (
    alt.Chart(monthly_jobtype_boro, title="(C) Borough × Job Type — Filings Heatmap (Linked)")
      .transform_filter(brushMonths)
      .transform_filter(job_pass_expr)
      .transform_filter(alt.FieldOneOfPredicate(field="Job Type", oneOf=top_jobs))
      .transform_filter("!length(data('boroughSel_store').values) || boroughSel")
      .transform_aggregate(filings="sum(filings)", groupby=["Borough","Job Type"])
      .mark_rect()
      .encode(
          x=alt.X("Job Type:N", title="Job Type", sort=top_jobs, axis=alt.Axis(labelAngle=-45)),
          y=alt.Y("Borough:N", title="Borough", sort=BORO_DOMAIN),
          color=alt.Color("filings:Q", title="Filings",
                          scale=alt.Scale(type="log", domainMin=1, scheme="blues")),
          tooltip=[alt.Tooltip("Borough:N"), alt.Tooltip("Job Type:N"),
                   alt.Tooltip("filings:Q", title="Filings", format=",")]
      )
      .properties(width=440, height=220)
)

task1_viz1 = alt.vconcat(
    bars,
    alt.hconcat(worktype_mix, heatmap_C)
).resolve_scale(color="independent")

# ==================== TASK 1 Viz 3 (Butterfly + Monthly Diverging) ====================
df3 = DF[["Borough","Job Type","Filing Date",
          "Existing Dwelling Units","Proposed Dwelling Units","mmYYYY","month_start"]].copy()
df3["Existing Dwelling Units"] = pd.to_numeric(df3["Existing Dwelling Units"], errors="coerce")
df3["Proposed Dwelling Units"] = pd.to_numeric(df3["Proposed Dwelling Units"], errors="coerce")
df3 = df3.dropna(subset=["Existing Dwelling Units","Proposed Dwelling Units"])
df3["delta_units"] = df3["Proposed Dwelling Units"] - df3["Existing Dwelling Units"]

butter_base = (
    alt.Chart(df3)
      .transform_filter(job_pass_expr)
      .transform_filter(brushMonths)
      .transform_filter("!length(data('boroughSel_store').values) || boroughSel")
      .transform_calculate(
          adds_pos="datum.delta_units > 0 ? datum.delta_units : 0",
          rem_pos ="datum.delta_units < 0 ? -datum.delta_units : 0"
      )
      .transform_aggregate(adds="sum(adds_pos)", removals="sum(rem_pos)", groupby=["Borough"])
      .transform_calculate(total_activity="datum.adds + datum.removals")
      .transform_fold(["adds","removals"], as_=["kind","value"])
      .transform_calculate(
          signed_value="datum.kind == 'adds' ? datum.value : -datum.value",
          kind_label ="datum.kind == 'adds' ? 'Adds' : 'Removals'"
      )
)

zero_rule = alt.Chart(pd.DataFrame({"x":[0]})).mark_rule(color="#888").encode(x="x:Q")

butter_bars = (
    butter_base.mark_bar()
      .encode(
          y=alt.Y("Borough:N", sort=alt.SortField(field="total_activity", order="descending")),
          x=alt.X("signed_value:Q", title="Dwelling Units (adds right, removals left)", axis=alt.Axis(format="~s")),
          color=alt.Color("kind_label:N", scale=alt.Scale(domain=["Adds","Removals"])),
          tooltip=[
              alt.Tooltip("Borough:N"),
              alt.Tooltip("kind_label:N", title="Kind"),
              alt.Tooltip("value:Q", title="Units", format=",.0f")
          ]
      )
      .properties(width=900, height=220, title="(D) Dwelling Units — Adds vs Removals (Linked)")
)

monthly_addrem = (
    df3.groupby(["mmYYYY","month_start","Borough","Job Type"], observed=False)[["delta_units"]]
      .sum().reset_index()
)
monthly_long = (
    monthly_addrem.assign(
        adds=lambda d: d["delta_units"].clip(lower=0),
        removals=lambda d: (-d["delta_units"].clip(upper=0))
    ).melt(id_vars=["mmYYYY","month_start","Borough","Job Type"],
           value_vars=["adds","removals"], var_name="kind", value_name="units")
)
monthly_long["kind_label"] = monthly_long["kind"].map({"adds":"Adds","removals":"Removals"})
monthly_long["signed_units"] = monthly_long.apply(
    lambda r: r["units"] if r["kind"]=="adds" else -r["units"], axis=1
)

monthly_diverging = (
    alt.Chart(monthly_long, title="(E) Monthly Adds (up) & Removals (down) — Diverging Bars (Linked)")
      .transform_filter(job_pass_expr)
      .transform_filter(brushMonths)
      .transform_filter("!length(data('boroughSel_store').values) || boroughSel")
      .mark_bar()
      .encode(
          x=alt.X("yearmonth(month_start):T", title="Month", axis=alt.Axis(format="%b %y")),
          y=alt.Y("signed_units:Q", title="Dwelling Units"),
          xOffset=alt.XOffset("Borough:N"),
          color=alt.Color("Borough:N", scale=alt.Scale(scheme="category10")),
          tooltip=[alt.Tooltip("yearmonth(month_start):T", title="Month", format="%b %Y"),
                   alt.Tooltip("Borough:N"),
                   alt.Tooltip("kind_label:N", title="Type"),
                   alt.Tooltip("units:Q", title="Units", format=",.0f")]
      )
      .properties(width=900, height=220)
)

task1_viz3 = alt.vconcat(butter_bars, monthly_diverging).resolve_scale(color="independent")

# ==================== TASK 1 Viz 4 (Ridgeline + Top-10) ====================
E = DF.loc[DF["Filing Date"].notna()].copy()
eff = E.get("Approved Date", pd.Series(pd.NaT, index=E.index))
eff = eff.combine_first(E.get("First Permit Date", pd.Series(pd.NaT, index=E.index)))
eff = eff.combine_first(E.get("Signoff Date", pd.Series(pd.NaT, index=E.index)))
E["effective_approval_date"] = eff
E["approval_days"] = (E["effective_approval_date"] - E["Filing Date"]).dt.days
E = E.loc[E["approval_days"].notna() & (E["approval_days"] >= 0),
          ["Filing Date","approval_days","Borough","Job Type","mmYYYY","month_start"]].copy()
vals = E["approval_days"].astype(float).to_numpy()
p99 = np.nanpercentile(vals, 99) if vals.size else 180
cap_days = int(min(np.ceil(p99/30.0)*30.0, 180))
E["approval_days_cap"] = E["approval_days"].clip(0, cap_days)

ridge_job_sel = alt.selection_point(fields=["Job Type"], bind="legend", empty="all")

# --- Ridgeline WITHOUT facet: build one row per Borough and vconcat them ---
def ridge_row(boro):
    return (
        alt.Chart(E)
          .transform_filter(f"datum.Borough == '{boro}'")
          .transform_filter("!length(data('boroughSel_store').values) || boroughSel")
          .transform_filter(job_pass_expr)
          .transform_filter(brushMonths)
          .transform_density(
              density="approval_days_cap", groupby=["Job Type"],
              as_=["approval_days_cap","density"], extent=[0, float(cap_days)],
              steps=200, counts=False
          )
          .mark_area(opacity=0.7)
          .encode(
              x=alt.X("approval_days_cap:Q",
                      title=f"Approval time (days, capped at {cap_days})",
                      scale=alt.Scale(domain=[0, float(cap_days)])),
              y=alt.Y("density:Q", title=None),
              color=alt.Color("Job Type:N",
                              legend=alt.Legend(title="Job Type (ridgeline filter)")),
              tooltip=[
                  alt.Tooltip("Borough:N"), alt.Tooltip("Job Type:N"),
                  alt.Tooltip("approval_days_cap:Q", title="Days", format=",.0f"),
                  alt.Tooltip("density:Q", title="Density", format=".3f")
              ]
          )
          .properties(width=900, height=110, title=boro)
    )

rows = [ridge_row(b) for b in BORO_DOMAIN]
# add legend/param once (affects all ridgeline rows)
rows[0] = rows[0].add_params(ridge_job_sel)

ridge = alt.vconcat(*rows).resolve_scale(y="independent").properties(
    title="(F) Approval-time Ridgeline by Borough — Linked"
)

ranked = (
    alt.Chart(E)
      .transform_filter("!length(data('boroughSel_store').values) || boroughSel")
      .transform_filter(brushMonths)
      .transform_aggregate(n="count()", groupby=["Job Type"])
      .transform_joinaggregate(total="sum(n)")
      .transform_calculate(share="datum.n / datum.total")
      .transform_window(rank="rank(n)", sort=[alt.SortField("n", order="descending")])
      .transform_filter("datum.rank <= 10")
      .mark_bar()
      .encode(
          y=alt.Y("Job Type:N", sort="-x", title="Job Type"),
          x=alt.X("n:Q", title="Applications in selection"),
          color=alt.Color("Job Type:N", legend=None),
          tooltip=[alt.Tooltip("Job Type:N"), alt.Tooltip("n:Q", title="Count"),
                   alt.Tooltip("share:Q", title="Share", format=".1%")]
      )
      .properties(width=900, height=320, title="(G) Top-10 Job Types in the linked selection")
)

# ====================== FINAL LAYOUT (unchanged above) ======================
top_row = alt.hconcat(map_comp, ts_comp)
mid_row = task1_viz1
bot_row = task1_viz3
task3_main_dashboard = alt.vconcat(top_row, mid_row, bot_row)

# Build the ridgeline block (already defined as `ridge` + `ranked`)
ridge_block = alt.concat(ridge, ranked, spacing=8)

# ---- FINAL, single figure (generic concat; no width/height on vconcat) ----
final_task3 = (
    alt.concat(
        task3_main_dashboard,
        ridge_block,
        spacing=24
    )
    .resolve_scale(color='independent', size='independent')
    .add_params(jobSel)            # expose Job Type param globally
    .configure_view(stroke=None)
)

final_task3



Output hidden; open in https://colab.research.google.com to view.

In [19]:
# ========================== TASK 3: SPATIAL + NON-SPATIAL DASHBOARD ==========================
# Paste this cell as-is in Jupyter/Colab. Adjust the GeoJSON path and run.

import altair as alt, pandas as pd, numpy as np, geopandas as gpd
from pathlib import Path

# --------------------------------- Altair setup ---------------------------------
alt.data_transformers.disable_max_rows()
alt.renderers.enable("default")

# --------------------------------- Inputs ---------------------------------
DF = FINAL_DF.copy()  # Must exist in memory

# Prefer Drive path if present, else local
p_colab = Path("/content/drive/MyDrive/CS424-Assignment3/nyc_nta.geojson")
p_local = Path("/mnt/data/nyc_nta.geojson")
NTA_GEOJSON_PATH = p_colab if p_colab.exists() else p_local

# --------------------------------- Common Prep ---------------------------------
for c in ["Filing Date","Approved Date","First Permit Date","Signoff Date"]:
    if c in DF.columns:
        DF[c] = pd.to_datetime(DF[c], errors="coerce")

DF = DF.dropna(subset=["Filing Date","Borough","Job Type","NTA","Job Filing Number"]).copy()
DF["year"]        = DF["Filing Date"].dt.year.astype(int)
DF["month_num"]   = DF["Filing Date"].dt.month.astype(int)
DF["month_lbl"]   = DF["Filing Date"].dt.strftime("%b")
DF["month_start"] = DF["Filing Date"].values.astype("datetime64[M]")
DF["mmYYYY"]      = DF["month_start"].dt.strftime("%Y-%m")
DF["NTA_key"]     = DF["NTA"].astype("string").str.strip().str.upper()

# Borough normalization
boro_map = {
    "bronx":"Bronx","brooklyn":"Brooklyn","bk":"Brooklyn",
    "manhattan":"Manhattan","mn":"Manhattan",
    "queens":"Queens","qn":"Queens",
    "staten island":"Staten Island","si":"Staten Island"
}
DF["Borough"] = DF["Borough"].astype(str).str.strip().map(lambda s: boro_map.get(s.lower(), s.title()))
BORO_DOMAIN = ["Manhattan","Brooklyn","Queens","Bronx","Staten Island"]

# Map NTA -> dominant Borough (from DF)
nta_boro = (
    DF[["NTA_key","Borough"]]
    .groupby("NTA_key", observed=False)["Borough"]
    .agg(lambda s: s.value_counts().idxmax())
    .reset_index()
)

# --------------------------------- Polygons & Choropleth data ---------------------------------
gpoly = gpd.read_file(NTA_GEOJSON_PATH)
candidate_cols = [
    'NTAName','ntaName','ntaname',
    'NTA','nta',
    'NTA2020','NTA2020_NAME','NTAName20',
    'NTACode','NTACode20','NTACode_20',
    'ntacode','ntacode20'
]
poly_key_col = next((c for c in candidate_cols if c in gpoly.columns), None)
if poly_key_col is None:
    raise ValueError(f"No NTA name/code column found in polygons. Columns: {list(gpoly.columns)}")

gpoly = gpoly.copy()
gpoly["NTA_key"]   = gpoly[poly_key_col].astype("string").str.strip().str.upper()
gpoly["NTA_label"] = gpoly[poly_key_col].astype("string")

# 2021–2025 totals for choropleth color (tweak years if needed)
yrs = DF["month_start"].dt.year
nta_totals = (
    DF.loc[(yrs >= 2021) & (yrs <= 2025)]
      .groupby("NTA_key", observed=False)["Job Filing Number"]
      .nunique().rename("filings_total").reset_index()
)
gpoly2 = gpoly.merge(nta_totals, on="NTA_key", how="left").fillna({"filings_total":0})

# Ensure WGS84 and compute robust centroids (projected -> back to WGS84)
if gpoly2.crs is None or (getattr(gpoly2.crs,"to_epsg",lambda:None)() != 4326):
    gpoly2 = gpoly2.to_crs(epsg=4326)
gpoly_proj = gpoly2.to_crs(epsg=2263)  # NY Long Island ftUS
cent_geom_proj = gpoly_proj.geometry.centroid
cent_wgs = gpd.GeoSeries(cent_geom_proj, crs=2263).to_crs(epsg=4326)
centroids_df = pd.DataFrame({"NTA_key": gpoly2["NTA_key"].values, "lon": cent_wgs.x.values, "lat": cent_wgs.y.values})

# Monthly tables
nta_monthly = (
    DF.groupby(["NTA_key","Job Type","month_start","mmYYYY"], observed=False)["Job Filing Number"]
      .nunique().rename("filings").reset_index()
)
city_monthly = (
    DF.groupby(["month_start","mmYYYY","Job Type"], observed=False)["Job Filing Number"]
      .nunique().rename("filings_city").reset_index()
)
month_order = (
    city_monthly[["month_start","mmYYYY"]]
    .drop_duplicates()
    .sort_values("month_start")["mmYYYY"].tolist()
)

# --------------------------------- Shared Controls ---------------------------------
jobtype_opts = ["(All)"] + sorted(DF["Job Type"].dropna().astype(str).unique().tolist())
jobSel      = alt.param("jobSel", value="(All)", bind=alt.binding_select(options=jobtype_opts, name="Job Type"))
brushMonths = alt.selection_interval(name="brushMonths", encodings=["x"], empty=True)

# Two selections:
boroughSel  = alt.selection_point(name="boroughSel", fields=["Borough"], toggle=True, empty=True)
ntaSel      = alt.selection_point(name="ntaSel", fields=["NTA_key"], on="click", clear="dblclick")

job_pass_expr = "(isValid(jobSel) && jobSel != '(All)') ? datum['Job Type'] == jobSel : true"

# --------------------------------- MAP (choropleth + bubbles) ---------------------------------
gjson = gpoly2.to_json()

choropleth = (
    alt.Chart(alt.Data(values=gjson, format={'type':'json','property':'features'}))
      .transform_calculate(NTA_key="datum.properties.NTA_key")
      .transform_lookup(lookup="NTA_key", from_=alt.LookupData(nta_boro, key="NTA_key", fields=["Borough"]))
      .mark_geoshape(stroke='black', strokeWidth=0.25)
      .encode(
          color=alt.Color('properties.filings_total:Q', title='Total filings 2021–2025',
                          scale=alt.Scale(type='symlog', scheme='viridis')),
          tooltip=[alt.Tooltip(f'properties.{poly_key_col}:N', title='NTA'),
                   alt.Tooltip('Borough:N', title='Borough'),
                   alt.Tooltip('properties.filings_total:Q', title='Total filings', format=',')],
          opacity=alt.condition(ntaSel, alt.value(0.95), alt.value(0.75))
      )
      .properties(width=520, height=620, title="Filings by NTA (symlog)")
)

bubbles = (
    alt.Chart(nta_monthly)
      .transform_filter(job_pass_expr)
      .transform_filter(brushMonths)  # aggregate only inside brushed months
      .transform_aggregate(total_filings="sum(filings)", groupby=["NTA_key"])
      .transform_lookup(lookup="NTA_key", from_=alt.LookupData(centroids_df, key="NTA_key", fields=["lon","lat"]))
      .transform_lookup(lookup="NTA_key", from_=alt.LookupData(nta_boro, key="NTA_key", fields=["Borough"]))
      .mark_circle(stroke="black", strokeWidth=0.5)
      .encode(
          longitude="lon:Q", latitude="lat:Q",
          size=alt.Size("total_filings:Q", title="Filings (in brush)", scale=alt.Scale(range=[10, 1200])),
          color=alt.Color("total_filings:Q", title="Filings (in brush)", scale=alt.Scale(scheme="plasma")),
          tooltip=[alt.Tooltip("Borough:N"), alt.Tooltip("total_filings:Q", title="Filings (window)", format=",")],
          opacity=alt.condition(ntaSel, alt.value(1.0), alt.value(0.85))
      )
      .properties(width=520, height=620)
)

# Attach BOTH selections ONCE to the parent layer (so clicks on either layer drive them)
map_comp = alt.layer(choropleth, bubbles).add_params(ntaSel, boroughSel).resolve_scale(color="independent", size="independent")

# --------------------------------- TIME SERIES (linked) ---------------------------------
ts_city = (
    alt.Chart(city_monthly)
      .transform_filter(job_pass_expr)
      .transform_aggregate(filings_city="sum(filings_city)", groupby=["mmYYYY","month_start"])
      .mark_area(opacity=0.25)
      .encode(
          x=alt.X("mmYYYY:N", sort=month_order, title="Month (brush to set map window)"),
          y=alt.Y("filings_city:Q", title="Citywide filings"),
          tooltip=[alt.Tooltip("mmYYYY:N", title="Month"),
                   alt.Tooltip("filings_city:Q", title="Citywide", format=",")]
      )
      .add_params(brushMonths)
      .properties(width=900, height=120)
)

# NTA-level line that responds to BOTH map selection and the brush
ts_sel = (
    alt.Chart(nta_monthly)
      .transform_filter(ntaSel)        # NTA selected on the map (bubble or polygon)
      .transform_filter(job_pass_expr)
      .transform_filter(brushMonths)   # reflect brushed months
      .transform_aggregate(filings="sum(filings)", groupby=["mmYYYY","month_start"])
      .mark_line(point=True)
      .encode(
          x=alt.X("mmYYYY:N", sort=month_order, title="Month"),
          y=alt.Y("filings:Q", title="Filings (selected NTA)"),
          tooltip=[alt.Tooltip("mmYYYY:N", title="Month"),
                   alt.Tooltip("filings:Q", title="Filings", format=",")]
      )
      .properties(width=900, height=180)
)

ts_title = (
    alt.Chart(pd.DataFrame({'t': ['Monthly filings — click a bubble/polygon; brush to aggregate bubbles/line']}))
      .mark_text(dy=-5, fontSize=13, fontWeight='bold', align='left')
      .encode(text='t:N')
      .properties(width=900, height=20)
)

ts_comp = alt.vconcat(ts_title, ts_city, ts_sel).resolve_scale(x="shared")

# --------------------------------- TASK 1 Viz 1 (A/B/C) ---------------------------------
m_job = (
    DF.groupby(["year","month_num","month_lbl","mmYYYY","Job Type","Borough"], observed=False)
      .size().reset_index(name="filings")
)
topN = 6
top_jobs = (
    m_job.groupby("Job Type", observed=False)["filings"].sum()
         .sort_values(ascending=False).head(topN).index.tolist()
)

legend_sel = alt.selection_point(fields=["Job Type"], bind="legend", toggle=True, empty=True)

axis_month = alt.Axis(
    title="Month", labelAngle=-40,
    labelExpr='["","Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"][toNumber(datum.value)]'
)

bars = (
    alt.Chart(m_job)
      .transform_filter(brushMonths)
      .transform_filter(legend_sel)
      .transform_filter(boroughSel)  # <-- selection predicate (replaces *_store string)
      .transform_aggregate(filings="sum(filings)", groupby=["year","month_num","month_lbl","Job Type","mmYYYY"])
      .mark_bar()
      .encode(
          x=alt.X("month_num:O", sort=list(range(1,13)), axis=axis_month),
          y=alt.Y("filings:Q", title="Filings (count)"),
          color=alt.Color("Job Type:N", title="Job Type", scale=alt.Scale(domain=top_jobs)),
          xOffset=alt.XOffset("Job Type:N"),
          order=alt.Order("Job Type:N"),
          tooltip=["month_lbl:N","Job Type:N",alt.Tooltip("filings:Q", title="Filings")]
      )
      .add_params(legend_sel)
      .properties(title="(A) Monthly Filings by Job Type — Linked to Brush & Borough Selection",
                  width=900, height=320)
)

# Work-type mix (robust boolean parsing)
worktype_cols = [c for c in DF.columns if c.endswith("( Work Type)") or c.endswith("(Work Type)")]
def to_bool_series(s):
    if s.dtype == bool: return s.fillna(False)
    if pd.api.types.is_numeric_dtype(s): return s.fillna(0).astype(int).astype(bool)
    sv = s.astype(str).str.strip().str.lower()
    return sv.isin(["y","yes","true","t","1"])
base_wt = DF[["Borough","Job Type","mmYYYY"] + worktype_cols].copy() if worktype_cols else pd.DataFrame(columns=["Borough","Job Type","mmYYYY"])
for c in worktype_cols:
    base_wt[c] = to_bool_series(base_wt[c])
longWT = (
    base_wt.melt(id_vars=["Borough","Job Type","mmYYYY"], value_vars=worktype_cols,
                 var_name="WorkTypeRaw", value_name="has_work")
    if not base_wt.empty else pd.DataFrame(columns=["Borough","Job Type","mmYYYY","WorkTypeRaw","has_work"])
)
longWT = longWT[longWT["has_work"]].copy() if not longWT.empty else longWT
if not longWT.empty:
    longWT["WorkType"] = longWT["WorkTypeRaw"].str.replace(r"\s*\( ?Work Type ?\)\s*", "", regex=True)

worktype_mix = (
    alt.Chart(longWT if not longWT.empty else pd.DataFrame(columns=["Borough","WorkType"]))
      .transform_filter(brushMonths)
      .transform_filter(job_pass_expr)
      .transform_filter(boroughSel)  # <-- selection predicate
      .mark_bar()
      .encode(
          y=alt.Y("Borough:N", sort=BORO_DOMAIN, title="Borough"),
          x=alt.X("count():Q", stack="normalize", title="Share of filings"),
          color=alt.Color("WorkType:N", title="Work Type"),
          tooltip=[alt.Tooltip("Borough:N"), alt.Tooltip("WorkType:N", title="Work Type"),
                   alt.Tooltip("count():Q", title="Filings", format=",.0f")]
      )
      .properties(title="(B) Work-Type Composition by Borough — Linked", width=440, height=220)
)

monthly_jobtype_boro = (
    DF.groupby(["mmYYYY","Borough","Job Type"], observed=False)
      .size().reset_index(name="filings")
)
heatmap_C = (
    alt.Chart(monthly_jobtype_boro, title="(C) Borough × Job Type — Filings Heatmap (Linked)")
      .transform_filter(brushMonths)
      .transform_filter(job_pass_expr)
      .transform_filter(alt.FieldOneOfPredicate(field="Job Type", oneOf=top_jobs))
      .transform_filter(boroughSel)  # <-- selection predicate
      .transform_aggregate(filings="sum(filings)", groupby=["Borough","Job Type"])
      .mark_rect()
      .encode(
          x=alt.X("Job Type:N", title="Job Type", sort=top_jobs, axis=alt.Axis(labelAngle=-45)),
          y=alt.Y("Borough:N", title="Borough", sort=BORO_DOMAIN),
          color=alt.Color("filings:Q", title="Filings", scale=alt.Scale(type="log", domainMin=1, scheme="blues")),
          tooltip=[alt.Tooltip("Borough:N"), alt.Tooltip("Job Type:N"),
                   alt.Tooltip("filings:Q", title="Filings", format=",")]
      )
      .properties(width=440, height=220)
)

task1_viz1 = alt.vconcat(bars, alt.hconcat(worktype_mix, heatmap_C)).resolve_scale(color="independent")

# --------------------------------- TASK 1 Viz 3 (Butterfly + monthly diverging) ---------------------------------
df3 = DF[["Borough","Job Type","Filing Date","Existing Dwelling Units","Proposed Dwelling Units","mmYYYY","month_start"]].copy()
df3["Existing Dwelling Units"] = pd.to_numeric(df3["Existing Dwelling Units"], errors="coerce")
df3["Proposed Dwelling Units"] = pd.to_numeric(df3["Proposed Dwelling Units"], errors="coerce")
df3 = df3.dropna(subset=["Existing Dwelling Units","Proposed Dwelling Units"])
df3["delta_units"] = df3["Proposed Dwelling Units"] - df3["Existing Dwelling Units"]

butter_base = (
    alt.Chart(df3)
      .transform_filter(job_pass_expr)
      .transform_filter(brushMonths)
      .transform_filter(boroughSel)   # <-- selection predicate
      .transform_calculate(
          adds_pos="datum.delta_units > 0 ? datum.delta_units : 0",
          rem_pos ="datum.delta_units < 0 ? -datum.delta_units : 0"
      )
      .transform_aggregate(adds="sum(adds_pos)", removals="sum(rem_pos)", groupby=["Borough"])
      .transform_calculate(total_activity="datum.adds + datum.removals")
      .transform_fold(["adds","removals"], as_=["kind","value"])
      .transform_calculate(
          signed_value="datum.kind == 'adds' ? datum.value : -datum.value",
          kind_label ="datum.kind == 'adds' ? 'Adds' : 'Removals'"
      )
)

butter_bars = (
    butter_base.mark_bar()
      .encode(
          y=alt.Y("Borough:N", sort=alt.SortField(field="total_activity", order="descending")),
          x=alt.X("signed_value:Q", title="Dwelling Units (adds right, removals left)", axis=alt.Axis(format="~s")),
          color=alt.Color("kind_label:N", scale=alt.Scale(domain=["Adds","Removals"])),
          tooltip=[alt.Tooltip("Borough:N"), alt.Tooltip("kind_label:N", title="Kind"),
                   alt.Tooltip("value:Q", title="Units", format=",.0f")]
      )
      .properties(width=900, height=220, title="(D) Dwelling Units — Adds vs Removals (Linked)")
)

monthly_addrem = df3.groupby(["mmYYYY","month_start","Borough","Job Type"], observed=False)[["delta_units"]].sum().reset_index()
monthly_long = (
    monthly_addrem.assign(adds=lambda d: d["delta_units"].clip(lower=0),
                          removals=lambda d: (-d["delta_units"].clip(upper=0)))
    .melt(id_vars=["mmYYYY","month_start","Borough","Job Type"],
          value_vars=["adds","removals"], var_name="kind", value_name="units")
)
monthly_long["kind_label"] = monthly_long["kind"].map({"adds":"Adds","removals":"Removals"})
monthly_long["signed_units"] = np.where(monthly_long["kind"]=="adds", monthly_long["units"], -monthly_long["units"])

monthly_diverging = (
    alt.Chart(monthly_long, title="(E) Monthly Adds (up) & Removals (down) — Diverging Bars (Linked)")
      .transform_filter(job_pass_expr)
      .transform_filter(brushMonths)
      .transform_filter(boroughSel)  # <-- selection predicate
      .mark_bar()
      .encode(
          x=alt.X("yearmonth(month_start):T", title="Month", axis=alt.Axis(format="%b %y")),
          y=alt.Y("signed_units:Q", title="Dwelling Units"),
          xOffset=alt.XOffset("Borough:N"),
          color=alt.Color("Borough:N", scale=alt.Scale(scheme="category10")),
          tooltip=[alt.Tooltip("yearmonth(month_start):T", title="Month", format="%b %Y"),
                   alt.Tooltip("Borough:N"),
                   alt.Tooltip("kind_label:N", title="Type"),
                   alt.Tooltip("units:Q", title="Units", format=",.0f")]
      )
      .properties(width=900, height=220)
)

task1_viz3 = alt.vconcat(butter_bars, monthly_diverging).resolve_scale(color="independent")

# --------------------------------- TASK 1 Viz 4 (Ridgeline + Top-10) ---------------------------------
E = DF.loc[DF["Filing Date"].notna()].copy()
eff = E.get("Approved Date", pd.Series(pd.NaT, index=E.index))
eff = eff.combine_first(E.get("First Permit Date", pd.Series(pd.NaT, index=E.index)))
eff = eff.combine_first(E.get("Signoff Date", pd.Series(pd.NaT, index=E.index)))
E["effective_approval_date"] = eff
E["approval_days"] = (E["effective_approval_date"] - E["Filing Date"]).dt.days
E = E.loc[E["approval_days"].notna() & (E["approval_days"] >= 0),
          ["Filing Date","approval_days","Borough","Job Type","mmYYYY","month_start"]].copy()
vals = E["approval_days"].astype(float).to_numpy()
p99 = np.nanpercentile(vals, 99) if vals.size else 180
cap_days = int(min(np.ceil(p99/30.0)*30.0, 180))
E["approval_days_cap"] = E["approval_days"].clip(0, cap_days)

ridge_job_sel = alt.selection_point(fields=["Job Type"], bind="legend", empty="all")

def ridge_row(boro):
    return (
        alt.Chart(E)
          .transform_filter(f"datum.Borough == '{boro}'")
          .transform_filter(boroughSel)   # <-- selection predicate
          .transform_filter(job_pass_expr)
          .transform_filter(brushMonths)
          .transform_density(
              density="approval_days_cap", groupby=["Job Type"],
              as_=["approval_days_cap","density"], extent=[0, float(cap_days)], steps=200, counts=False
          )
          .mark_area(opacity=0.7)
          .encode(
              x=alt.X("approval_days_cap:Q",
                      title=f"Approval time (days, capped at {cap_days})",
                      scale=alt.Scale(domain=[0, float(cap_days)])),
              y=alt.Y("density:Q", title=None),
              color=alt.Color("Job Type:N", legend=alt.Legend(title="Job Type (ridgeline filter)")),
              tooltip=[alt.Tooltip("Borough:N"), alt.Tooltip("Job Type:N"),
                       alt.Tooltip("approval_days_cap:Q", title="Days", format=",.0f"),
                       alt.Tooltip("density:Q", title="Density", format=".3f")]
          )
          .properties(width=900, height=110, title=boro)
    )

rows = [ridge_row(b) for b in BORO_DOMAIN]
rows[0] = rows[0].add_params(ridge_job_sel)  # add legend/param once
ridge = alt.vconcat(*rows).resolve_scale(y="independent").properties(
    title="(F) Approval-time Ridgeline by Borough — Linked"
)

ranked = (
    alt.Chart(E)
      .transform_filter(boroughSel)   # <-- selection predicate
      .transform_filter(brushMonths)
      .transform_aggregate(n="count()", groupby=["Job Type"])
      .transform_joinaggregate(total="sum(n)")
      .transform_calculate(share="datum.n / datum.total")
      .transform_window(rank="rank(n)", sort=[alt.SortField("n", order="descending")])
      .transform_filter("datum.rank <= 10")
      .mark_bar()
      .encode(
          y=alt.Y("Job Type:N", sort="-x", title="Job Type"),
          x=alt.X("n:Q", title="Applications in selection"),
          color=alt.Color("Job Type:N", legend=None),
          tooltip=[alt.Tooltip("Job Type:N"), alt.Tooltip("n:Q", title="Count"),
                   alt.Tooltip("share:Q", title="Share", format=".1%")]
      )
      .properties(width=900, height=320, title="(G) Top-10 Job Types in the linked selection")
)

# --------------------------------- Final Layout ---------------------------------
top_row = alt.hconcat(map_comp, ts_comp)
mid_row = task1_viz1
bot_row = task1_viz3
task3_main_dashboard = alt.vconcat(top_row, mid_row, bot_row)

ridge_block = alt.concat(ridge, ranked, spacing=8)

final_task3 = (
    alt.concat(task3_main_dashboard, ridge_block, spacing=24)
      .resolve_scale(color='independent', size='independent')
      .add_params(jobSel)            # expose Job Type param globally
      .configure_view(stroke=None)
)

final_task3


Output hidden; open in https://colab.research.google.com to view.

In [ ]:
# ========================== TASK 3: SPATIAL + NON-SPATIAL DASHBOARD ==========================
# Paste this cell as-is in Jupyter/Colab. Adjust the GeoJSON path and run.

import altair as alt, pandas as pd, numpy as np, geopandas as gpd
from pathlib import Path

# --------------------------------- Altair setup ---------------------------------
alt.data_transformers.disable_max_rows()
alt.renderers.enable("default")

# --------------------------------- Inputs ---------------------------------
DF = FINAL_DF.copy()  # Must exist in memory

# Prefer Drive path if present, else local
p_colab = Path("/content/drive/MyDrive/CS424-Assignment3/nyc_nta.geojson")
p_local = Path("/mnt/data/nyc_nta.geojson")
NTA_GEOJSON_PATH = p_colab if p_colab.exists() else p_local

# --------------------------------- Common Prep ---------------------------------
for c in ["Filing Date","Approved Date","First Permit Date","Signoff Date"]:
    if c in DF.columns:
        DF[c] = pd.to_datetime(DF[c], errors="coerce")

DF = DF.dropna(subset=["Filing Date","Borough","Job Type","NTA","Job Filing Number"]).copy()
DF["year"]        = DF["Filing Date"].dt.year.astype(int)
DF["month_num"]   = DF["Filing Date"].dt.month.astype(int)
DF["month_lbl"]   = DF["Filing Date"].dt.strftime("%b")
DF["month_start"] = DF["Filing Date"].values.astype("datetime64[M]")
DF["mmYYYY"]      = DF["month_start"].dt.strftime("%Y-%m")
DF["NTA_key"]     = DF["NTA"].astype("string").str.strip().str.upper()

# Borough normalization
boro_map = {
    "bronx":"Bronx","brooklyn":"Brooklyn","bk":"Brooklyn",
    "manhattan":"Manhattan","mn":"Manhattan",
    "queens":"Queens","qn":"Queens",
    "staten island":"Staten Island","si":"Staten Island"
}
DF["Borough"] = DF["Borough"].astype(str).str.strip().map(lambda s: boro_map.get(s.lower(), s.title()))
BORO_DOMAIN = ["Manhattan","Brooklyn","Queens","Bronx","Staten Island"]

# Map NTA -> dominant Borough (from DF)
nta_boro = (
    DF[["NTA_key","Borough"]]
    .groupby("NTA_key", observed=False)["Borough"]
    .agg(lambda s: s.value_counts().idxmax())
    .reset_index()
)

# --------------------------------- Polygons & Choropleth data ---------------------------------
gpoly = gpd.read_file(NTA_GEOJSON_PATH)
candidate_cols = [
    'NTAName','ntaName','ntaname',
    'NTA','nta',
    'NTA2020','NTA2020_NAME','NTAName20',
    'NTACode','NTACode20','NTACode_20',
    'ntacode','ntacode20'
]
poly_key_col = next((c for c in candidate_cols if c in gpoly.columns), None)
if poly_key_col is None:
    raise ValueError(f"No NTA name/code column found in polygons. Columns: {list(gpoly.columns)}")

gpoly = gpoly.copy()
gpoly["NTA_key"]   = gpoly[poly_key_col].astype("string").str.strip().str.upper()
gpoly["NTA_label"] = gpoly[poly_key_col].astype("string")

# 2021–2025 totals for choropleth color (tweak years if needed)
yrs = DF["month_start"].dt.year
nta_totals = (
    DF.loc[(yrs >= 2021) & (yrs <= 2025)]
      .groupby("NTA_key", observed=False)["Job Filing Number"]
      .nunique().rename("filings_total").reset_index()
)
gpoly2 = gpoly.merge(nta_totals, on="NTA_key", how="left").fillna({"filings_total":0})

# Ensure WGS84 and compute robust centroids (projected -> back to WGS84)
if gpoly2.crs is None or (getattr(gpoly2.crs,"to_epsg",lambda:None)() != 4326):
    gpoly2 = gpoly2.to_crs(epsg=4326)
gpoly_proj = gpoly2.to_crs(epsg=2263)  # NY Long Island ftUS
cent_geom_proj = gpoly_proj.geometry.centroid
cent_wgs = gpd.GeoSeries(cent_geom_proj, crs=2263).to_crs(epsg=4326)
centroids_df = pd.DataFrame({"NTA_key": gpoly2["NTA_key"].values, "lon": cent_wgs.x.values, "lat": cent_wgs.y.values})

# Monthly tables
nta_monthly = (
    DF.groupby(["NTA_key","Job Type","month_start","mmYYYY"], observed=False)["Job Filing Number"]
      .nunique().rename("filings").reset_index()
)
city_monthly = (
    DF.groupby(["month_start","mmYYYY","Job Type"], observed=False)["Job Filing Number"]
      .nunique().rename("filings_city").reset_index()
)
month_order = (
    city_monthly[["month_start","mmYYYY"]]
    .drop_duplicates()
    .sort_values("month_start")["mmYYYY"].tolist()
)

# --------------------------------- Shared Controls ---------------------------------
jobtype_opts = ["(All)"] + sorted(DF["Job Type"].dropna().astype(str).unique().tolist())
jobSel     = alt.param("jobSel", value="(All)", bind=alt.binding_select(options=jobtype_opts, name="Job Type"))
brushMonths= alt.selection_interval(name="brushMonths", encodings=["x"], empty=True)
boroughSel = alt.selection_point(name="boroughSel", fields=["Borough"], empty=True)  # for non-spatial links
ntaSel     = alt.selection_point(name="ntaSel", fields=["NTA_key"], on="click", clear="dblclick")  # bubble/polygon
job_pass_expr = "(isValid(jobSel) && jobSel != '(All)') ? datum['Job Type'] == jobSel : true"

# --------------------------------- MAP (choropleth + bubbles) ---------------------------------
gjson = gpoly2.to_json()

choropleth = (
    alt.Chart(alt.Data(values=gjson, format={'type':'json','property':'features'}))
      .transform_calculate(NTA_key="datum.properties.NTA_key")
      .transform_lookup(lookup="NTA_key", from_=alt.LookupData(nta_boro, key="NTA_key", fields=["Borough"]))
      .mark_geoshape(stroke='black', strokeWidth=0.25)
      .encode(
          color=alt.Color('properties.filings_total:Q', title='Total filings 2021–2025',
                          scale=alt.Scale(type='symlog', scheme='viridis')),
          tooltip=[alt.Tooltip(f'properties.{poly_key_col}:N', title='NTA'),
                   alt.Tooltip('Borough:N', title='Borough'),
                   alt.Tooltip('properties.filings_total:Q', title='Total filings', format=',')],
          opacity=alt.condition(ntaSel, alt.value(0.95), alt.value(0.75))
      )
      .properties(width=520, height=620, title="Filings by NTA (symlog)")
)

bubbles = (
    alt.Chart(nta_monthly)
      .transform_filter(job_pass_expr)
      .transform_filter(brushMonths)  # aggregate only inside brushed months
      .transform_aggregate(total_filings="sum(filings)", groupby=["NTA_key"])
      .transform_lookup(lookup="NTA_key", from_=alt.LookupData(centroids_df, key="NTA_key", fields=["lon","lat"]))
      .transform_lookup(lookup="NTA_key", from_=alt.LookupData(nta_boro, key="NTA_key", fields=["Borough"]))
      .mark_circle(stroke="black", strokeWidth=0.5)
      .encode(
          longitude="lon:Q", latitude="lat:Q",
          size=alt.Size("total_filings:Q", title="Filings (in brush)", scale=alt.Scale(range=[10, 1200])),
          color=alt.Color("total_filings:Q", title="Filings (in brush)", scale=alt.Scale(scheme="plasma")),
          tooltip=[alt.Tooltip("Borough:N"), alt.Tooltip("total_filings:Q", title="Filings (window)", format=",")],
          opacity=alt.condition(ntaSel, alt.value(1.0), alt.value(0.85))
      )
      .properties(width=520, height=620)
)

# Attach ntaSel to the LAYER so clicks on geoshapes OR bubbles drive selection
map_comp = alt.layer(choropleth, bubbles).add_params(ntaSel).resolve_scale(color="independent", size="independent")

# --------------------------------- TIME SERIES (linked) ---------------------------------
ts_city = (
    alt.Chart(city_monthly)
      .transform_filter(job_pass_expr)
      .transform_aggregate(filings_city="sum(filings_city)", groupby=["mmYYYY","month_start"])
      .mark_area(opacity=0.25)
      .encode(
          x=alt.X("mmYYYY:N", sort=month_order, title="Month (brush to set map window)"),
          y=alt.Y("filings_city:Q", title="Citywide filings"),
          tooltip=[alt.Tooltip("mmYYYY:N", title="Month"),
                   alt.Tooltip("filings_city:Q", title="Citywide", format=",")]
      )
      .add_params(brushMonths)
      .properties(width=900, height=120)
)

ts_sel = (
    alt.Chart(nta_monthly)
      .transform_filter(ntaSel)  # <-- this is what makes bubble/polygon click update the line
      .transform_filter(job_pass_expr)
      .transform_aggregate(filings="sum(filings)", groupby=["mmYYYY","month_start"])
      .mark_line(point=True)
      .encode(
          x=alt.X("mmYYYY:N", sort=month_order, title="Month"),
          y=alt.Y("filings:Q", title="Filings (selected NTA)"),
          tooltip=[alt.Tooltip("mmYYYY:N", title="Month"),
                   alt.Tooltip("filings:Q", title="Filings", format=",")]
      )
      .properties(width=900, height=180)
)

ts_title = (
    alt.Chart(pd.DataFrame({'t': ['Monthly filings — click a bubble/polygon; brush to aggregate bubbles']}))
      .mark_text(dy=-5, fontSize=13, fontWeight='bold', align='left')
      .encode(text='t:N')
      .properties(width=900, height=20)
)

ts_comp = alt.vconcat(ts_title, ts_city, ts_sel).resolve_scale(x="shared")

# --------------------------------- TASK 1 Viz 1 (A/B/C) ---------------------------------
m_job = (
    DF.groupby(["year","month_num","month_lbl","mmYYYY","Job Type","Borough"], observed=False)
      .size().reset_index(name="filings")
)
topN = 6
top_jobs = (
    m_job.groupby("Job Type", observed=False)["filings"].sum()
         .sort_values(ascending=False).head(topN).index.tolist()
)

legend_sel = alt.selection_point(fields=["Job Type"], bind="legend", toggle=True, empty=True)

axis_month = alt.Axis(
    title="Month", labelAngle=-40,
    labelExpr='["","Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"][toNumber(datum.value)]'
)

bars = (
    alt.Chart(m_job)
      .transform_filter(brushMonths)
      .transform_filter(legend_sel)
      .transform_filter("!length(data('boroughSel_store').values) || boroughSel")
      .transform_aggregate(filings="sum(filings)", groupby=["year","month_num","month_lbl","Job Type","mmYYYY"])
      .mark_bar()
      .encode(
          x=alt.X("month_num:O", sort=list(range(1,13)), axis=axis_month),
          y=alt.Y("filings:Q", title="Filings (count)"),
          color=alt.Color("Job Type:N", title="Job Type", scale=alt.Scale(domain=top_jobs)),
          xOffset=alt.XOffset("Job Type:N"),
          order=alt.Order("Job Type:N"),
          tooltip=["month_lbl:N","Job Type:N",alt.Tooltip("filings:Q", title="Filings")]
      )
      .add_params(legend_sel)
      .properties(title="(A) Monthly Filings by Job Type — Linked to Brush & Borough Selection",
                  width=900, height=320)
)

# Work-type mix (robust boolean parsing)
worktype_cols = [c for c in DF.columns if c.endswith("( Work Type)") or c.endswith("(Work Type)")]
def to_bool_series(s):
    if s.dtype == bool: return s.fillna(False)
    if pd.api.types.is_numeric_dtype(s): return s.fillna(0).astype(int).astype(bool)
    sv = s.astype(str).str.strip().str.lower()
    return sv.isin(["y","yes","true","t","1"])
base_wt = DF[["Borough","Job Type","mmYYYY"] + worktype_cols].copy() if worktype_cols else pd.DataFrame(columns=["Borough","Job Type","mmYYYY"])
for c in worktype_cols:
    base_wt[c] = to_bool_series(base_wt[c])
longWT = (
    base_wt.melt(id_vars=["Borough","Job Type","mmYYYY"], value_vars=worktype_cols,
                 var_name="WorkTypeRaw", value_name="has_work")
    if not base_wt.empty else pd.DataFrame(columns=["Borough","Job Type","mmYYYY","WorkTypeRaw","has_work"])
)
longWT = longWT[longWT["has_work"]].copy() if not longWT.empty else longWT
if not longWT.empty:
    longWT["WorkType"] = longWT["WorkTypeRaw"].str.replace(r"\s*\( ?Work Type ?\)\s*", "", regex=True)

worktype_mix = (
    alt.Chart(longWT if not longWT.empty else pd.DataFrame(columns=["Borough","WorkType"]))
      .transform_filter(brushMonths)
      .transform_filter(job_pass_expr)
      .transform_filter("!length(data('boroughSel_store').values) || boroughSel")
      .mark_bar()
      .encode(
          y=alt.Y("Borough:N", sort=BORO_DOMAIN, title="Borough"),
          x=alt.X("count():Q", stack="normalize", title="Share of filings"),
          color=alt.Color("WorkType:N", title="Work Type"),
          tooltip=[alt.Tooltip("Borough:N"), alt.Tooltip("WorkType:N", title="Work Type"),
                   alt.Tooltip("count():Q", title="Filings", format=",.0f")]
      )
      .properties(title="(B) Work-Type Composition by Borough — Linked", width=440, height=220)
)

monthly_jobtype_boro = (
    DF.groupby(["mmYYYY","Borough","Job Type"], observed=False)
      .size().reset_index(name="filings")
)
heatmap_C = (
    alt.Chart(monthly_jobtype_boro, title="(C) Borough × Job Type — Filings Heatmap (Linked)")
      .transform_filter(brushMonths)
      .transform_filter(job_pass_expr)
      .transform_filter(alt.FieldOneOfPredicate(field="Job Type", oneOf=top_jobs))
      .transform_filter("!length(data('boroughSel_store').values) || boroughSel")
      .transform_aggregate(filings="sum(filings)", groupby=["Borough","Job Type"])
      .mark_rect()
      .encode(
          x=alt.X("Job Type:N", title="Job Type", sort=top_jobs, axis=alt.Axis(labelAngle=-45)),
          y=alt.Y("Borough:N", title="Borough", sort=BORO_DOMAIN),
          color=alt.Color("filings:Q", title="Filings", scale=alt.Scale(type="log", domainMin=1, scheme="blues")),
          tooltip=[alt.Tooltip("Borough:N"), alt.Tooltip("Job Type:N"),
                   alt.Tooltip("filings:Q", title="Filings", format=",")]
      )
      .properties(width=440, height=220)
)

task1_viz1 = alt.vconcat(bars, alt.hconcat(worktype_mix, heatmap_C)).resolve_scale(color="independent")

# --------------------------------- TASK 1 Viz 3 (Butterfly + monthly diverging) ---------------------------------
df3 = DF[["Borough","Job Type","Filing Date","Existing Dwelling Units","Proposed Dwelling Units","mmYYYY","month_start"]].copy()
df3["Existing Dwelling Units"] = pd.to_numeric(df3["Existing Dwelling Units"], errors="coerce")
df3["Proposed Dwelling Units"] = pd.to_numeric(df3["Proposed Dwelling Units"], errors="coerce")
df3 = df3.dropna(subset=["Existing Dwelling Units","Proposed Dwelling Units"])
df3["delta_units"] = df3["Proposed Dwelling Units"] - df3["Existing Dwelling Units"]

butter_base = (
    alt.Chart(df3)
      .transform_filter(job_pass_expr)
      .transform_filter(brushMonths)
      .transform_filter("!length(data('boroughSel_store').values) || boroughSel")
      .transform_calculate(
          adds_pos="datum.delta_units > 0 ? datum.delta_units : 0",
          rem_pos ="datum.delta_units < 0 ? -datum.delta_units : 0"
      )
      .transform_aggregate(adds="sum(adds_pos)", removals="sum(rem_pos)", groupby=["Borough"])
      .transform_calculate(total_activity="datum.adds + datum.removals")
      .transform_fold(["adds","removals"], as_=["kind","value"])
      .transform_calculate(
          signed_value="datum.kind == 'adds' ? datum.value : -datum.value",
          kind_label ="datum.kind == 'adds' ? 'Adds' : 'Removals'"
      )
)

butter_bars = (
    butter_base.mark_bar()
      .encode(
          y=alt.Y("Borough:N", sort=alt.SortField(field="total_activity", order="descending")),
          x=alt.X("signed_value:Q", title="Dwelling Units (adds right, removals left)", axis=alt.Axis(format="~s")),
          color=alt.Color("kind_label:N", scale=alt.Scale(domain=["Adds","Removals"])),
          tooltip=[alt.Tooltip("Borough:N"), alt.Tooltip("kind_label:N", title="Kind"),
                   alt.Tooltip("value:Q", title="Units", format=",.0f")]
      )
      .properties(width=900, height=220, title="(D) Dwelling Units — Adds vs Removals (Linked)")
)

monthly_addrem = df3.groupby(["mmYYYY","month_start","Borough","Job Type"], observed=False)[["delta_units"]].sum().reset_index()
monthly_long = (
    monthly_addrem.assign(adds=lambda d: d["delta_units"].clip(lower=0),
                          removals=lambda d: (-d["delta_units"].clip(upper=0)))
    .melt(id_vars=["mmYYYY","month_start","Borough","Job Type"],
          value_vars=["adds","removals"], var_name="kind", value_name="units")
)
monthly_long["kind_label"] = monthly_long["kind"].map({"adds":"Adds","removals":"Removals"})
monthly_long["signed_units"] = np.where(monthly_long["kind"]=="adds", monthly_long["units"], -monthly_long["units"])

monthly_diverging = (
    alt.Chart(monthly_long, title="(E) Monthly Adds (up) & Removals (down) — Diverging Bars (Linked)")
      .transform_filter(job_pass_expr)
      .transform_filter(brushMonths)
      .transform_filter("!length(data('boroughSel_store').values) || boroughSel")
      .mark_bar()
      .encode(
          x=alt.X("yearmonth(month_start):T", title="Month", axis=alt.Axis(format="%b %y")),
          y=alt.Y("signed_units:Q", title="Dwelling Units"),
          xOffset=alt.XOffset("Borough:N"),
          color=alt.Color("Borough:N", scale=alt.Scale(scheme="category10")),
          tooltip=[alt.Tooltip("yearmonth(month_start):T", title="Month", format="%b %Y"),
                   alt.Tooltip("Borough:N"),
                   alt.Tooltip("kind_label:N", title="Type"),
                   alt.Tooltip("units:Q", title="Units", format=",.0f")]
      )
      .properties(width=900, height=220)
)

task1_viz3 = alt.vconcat(butter_bars, monthly_diverging).resolve_scale(color="independent")

# --------------------------------- TASK 1 Viz 4 (Ridgeline + Top-10) ---------------------------------
E = DF.loc[DF["Filing Date"].notna()].copy()
eff = E.get("Approved Date", pd.Series(pd.NaT, index=E.index))
eff = eff.combine_first(E.get("First Permit Date", pd.Series(pd.NaT, index=E.index)))
eff = eff.combine_first(E.get("Signoff Date", pd.Series(pd.NaT, index=E.index)))
E["effective_approval_date"] = eff
E["approval_days"] = (E["effective_approval_date"] - E["Filing Date"]).dt.days
E = E.loc[E["approval_days"].notna() & (E["approval_days"] >= 0),
          ["Filing Date","approval_days","Borough","Job Type","mmYYYY","month_start"]].copy()
vals = E["approval_days"].astype(float).to_numpy()
p99 = np.nanpercentile(vals, 99) if vals.size else 180
cap_days = int(min(np.ceil(p99/30.0)*30.0, 180))
E["approval_days_cap"] = E["approval_days"].clip(0, cap_days)

ridge_job_sel = alt.selection_point(fields=["Job Type"], bind="legend", empty="all")

def ridge_row(boro):
    return (
        alt.Chart(E)
          .transform_filter(f"datum.Borough == '{boro}'")
          .transform_filter("!length(data('boroughSel_store').values) || boroughSel")
          .transform_filter(job_pass_expr)
          .transform_filter(brushMonths)
          .transform_density(
              density="approval_days_cap", groupby=["Job Type"],
              as_=["approval_days_cap","density"], extent=[0, float(cap_days)], steps=200, counts=False
          )
          .mark_area(opacity=0.7)
          .encode(
              x=alt.X("approval_days_cap:Q",
                      title=f"Approval time (days, capped at {cap_days})",
                      scale=alt.Scale(domain=[0, float(cap_days)])),
              y=alt.Y("density:Q", title=None),
              color=alt.Color("Job Type:N", legend=alt.Legend(title="Job Type (ridgeline filter)")),
              tooltip=[alt.Tooltip("Borough:N"), alt.Tooltip("Job Type:N"),
                       alt.Tooltip("approval_days_cap:Q", title="Days", format=",.0f"),
                       alt.Tooltip("density:Q", title="Density", format=".3f")]
          )
          .properties(width=900, height=110, title=boro)
    )

rows = [ridge_row(b) for b in BORO_DOMAIN]
rows[0] = rows[0].add_params(ridge_job_sel)  # add legend/param once
ridge = alt.vconcat(*rows).resolve_scale(y="independent").properties(
    title="(F) Approval-time Ridgeline by Borough — Linked"
)

ranked = (
    alt.Chart(E)
      .transform_filter("!length(data('boroughSel_store').values) || boroughSel")
      .transform_filter(brushMonths)
      .transform_aggregate(n="count()", groupby=["Job Type"])
      .transform_joinaggregate(total="sum(n)")
      .transform_calculate(share="datum.n / datum.total")
      .transform_window(rank="rank(n)", sort=[alt.SortField("n", order="descending")])
      .transform_filter("datum.rank <= 10")
      .mark_bar()
      .encode(
          y=alt.Y("Job Type:N", sort="-x", title="Job Type"),
          x=alt.X("n:Q", title="Applications in selection"),
          color=alt.Color("Job Type:N", legend=None),
          tooltip=[alt.Tooltip("Job Type:N"), alt.Tooltip("n:Q", title="Count"),
                   alt.Tooltip("share:Q", title="Share", format=".1%")]
      )
      .properties(width=900, height=320, title="(G) Top-10 Job Types in the linked selection")
)

# --------------------------------- Final Layout (no width on vconcat/concat roots) ---------------------------------
# ====================== FINAL LAYOUT (unchanged above) ======================
top_row = alt.hconcat(map_comp, ts_comp)
mid_row = task1_viz1
bot_row = task1_viz3
task3_main_dashboard = alt.vconcat(top_row, mid_row, bot_row)

# Build the ridgeline block (already defined as ridge + ranked)
ridge_block = alt.concat(ridge, ranked, spacing=8)

# ---- FINAL, single figure (generic concat; no width/height on vconcat) ----
final_task3 = (
    alt.concat(
        task3_main_dashboard,
        ridge_block,
        spacing=24
    )
    .resolve_scale(color='independent', size='independent')
    .add_params(jobSel)            # expose Job Type param globally
    .configure_view(stroke=None)
)

final_task3


Buffered data was truncated after reaching the output size limit.